In [7]:

#path and file handling
import pathlib
import nd2
import os
import tifffile

In [8]:
#SET path for ND2 and Tiff files

# set path to BioImageArchive data directory:
data_archive_path = pathlib.Path('/Volumes/ScientificData/Users/Giulia(botgiu00)/Papers/bottacin2026/BioImageArchive/')

# input and output paths relative to BioImageArchive location
nd2_path = data_archive_path / '3DBiofilms'
tiff_path = data_archive_path / '3DBiofilms'

#set file name of nd2 file (leave out .nd2)
file_names = ["5_wt_small","5_wt_medium","5_wt_big","5_wt_mixed"] 

# Set channel index (optional if there are multiple channels)
channel = 2
channel_name = 'phase'

channels = [(0, 'mcherry'), (1, 'gfp'), (2, 'phase')]

#set position number
pos = 1

In [9]:
#create file names
for file_name in file_names:


    data_file = nd2_path / f"{file_name}.nd2"
    f = nd2.ND2File(data_file)
    raw_data = f.to_dask()
    #see metadata https://pypi.org/project/nd2/
    print(f.metadata.channels)
    print(f.experiment)
    num_dimensions = raw_data.ndim
    print(f"Number of dimensions: {num_dimensions}")
    print(f.sizes)
    f.close()

    for channel, channel_name in channels:
        if raw_data.ndim == 3:
            # Shape: (C, Y, X) - select specific channel
            pos_data = raw_data[channel, :, :]  # ✅ Correct
        
        elif raw_data.ndim == 4:
            pos_data = raw_data[pos, channel, :, :]
            
        elif raw_data.ndim == 5:
            pos_data = raw_data[0, pos, channel, :, :]  # Fixed: use index 0, not slice 1:2

        # create the output TIFF file path
        output_path = tiff_path / f"{file_name}_pos{pos}_{channel_name}.tif"

        #remove old tif file if it exists
        if os.path.exists(output_path):
            os.remove(output_path)

        tifffile.imwrite(output_path, pos_data.compute())

[Channel(channel=ChannelMeta(name='mcherry', index=0, color=Color(r=255, g=42, b=0, a=1.0), emissionLambdaNm=630.0, excitationLambdaNm=575.0), loops=LoopIndices(NETimeLoop=None, TimeLoop=0, XYPosLoop=1, ZStackLoop=None), microscope=Microscope(objectiveMagnification=10.0, objectiveName='Achromat 10x Ph1 ADL', objectiveNumericalAperture=0.25, zoomMagnification=1.0, immersionRefractiveIndex=1.0, projectiveMagnification=None, pinholeDiameterUm=None, modalityFlags=['fluorescence', 'camera']), volume=Volume(axesCalibrated=(True, True, False), axesCalibration=(0.65, 0.65, 1.0), axesInterpretation=('distance', 'distance', 'distance'), bitsPerComponentInMemory=16, bitsPerComponentSignificant=16, cameraTransformationMatrix=(-0.9999616468822087, 0.00875812563400387, -0.00875812563400387, -0.9999616468822087), componentCount=1, componentDataType='unsigned', voxelCount=(2304, 2304, 1), componentMaxima=[0.0], componentMinima=[0.0], pixelToStageTransformationMatrix=None)), Channel(channel=ChannelMeta